# The Ultimate Dynamic Ensembler
Upload `train.csv`, `test.csv`, and ANY prediction CSVs you have (`xgb_oof.csv`, `knn_test.csv`, etc.). It will dynamically detect them!


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
y = train['Drafted']


In [ ]:
# Dynamically load all available predictions!
available_models = []
oof_dict = {}
test_dict = {}

for oof_file in glob.glob('*_oof.csv'):
    model_name = oof_file.replace('_oof.csv', '')
    test_file = f"{model_name}_test.csv"
    if os.path.exists(test_file):
        available_models.append(model_name)
        oof_dict[model_name] = pd.read_csv(oof_file).values.flatten()
        test_dict[model_name] = pd.read_csv(test_file).values.flatten()
        print(f"Loaded {model_name} predictions!")
        
if len(available_models) == 0:
    raise ValueError("No prediction CSVs found! Upload them to the sidebar.")


In [ ]:
# Build the Meta-Features
oof_master = np.column_stack([oof_dict[m] for m in available_models])
test_master = np.column_stack([test_dict[m] for m in available_models])

print(f"Ensembling the following {len(available_models)} models: {available_models}")


In [ ]:
# 1. Logistic Regression Stacker
meta_lr = LogisticRegression(C=1.0, max_iter=1000)
meta_lr.fit(oof_master, y)
lr_score = roc_auc_score(y, meta_lr.predict_proba(oof_master)[:, 1])
print(f"Logistic Regression Stack CV: {lr_score:.5f}")


In [ ]:
# 2. Greedy Hill Climbing Stacker
iterations = 100
ensemble_oof = np.zeros(len(y))
ensemble_test = np.zeros(len(test))

for i in range(iterations):
    best_auc, best_idx, best_oof = 0, 0, None
    for m_idx in range(oof_master.shape[1]):
        temp_oof = (ensemble_oof * i + oof_master[:, m_idx]) / (i + 1)
        temp_auc = roc_auc_score(y, temp_oof)
        if temp_auc > best_auc:
            best_auc, best_idx, best_oof = temp_auc, m_idx, temp_oof
    ensemble_oof = best_oof
    ensemble_test = (ensemble_test * i + test_master[:, best_idx]) / (i + 1)

v6_score = roc_auc_score(y, ensemble_oof)
print(f"Hill Climbing Stack CV: {v6_score:.5f}")


In [ ]:
# 3. Rank Averaging Stacker
from scipy.stats import rankdata
ensemble_rank_oof = np.zeros(len(y))
ensemble_rank_test = np.zeros(len(test))

for m_idx in range(oof_master.shape[1]):
    # Convert probabilities to ranks (0.0 to 1.0)
    ensemble_rank_oof += rankdata(oof_master[:, m_idx]) / len(y)
    ensemble_rank_test += rankdata(test_master[:, m_idx]) / len(test_master)

ensemble_rank_oof /= oof_master.shape[1]
ensemble_rank_test /= test_master.shape[1]

rank_score = roc_auc_score(y, ensemble_rank_oof)
print(f"Rank Averaging Stack CV: {rank_score:.5f}")


In [ ]:
# 4. AutoGluon Non-Linear Stacker
# WARNING: This requires you to run '!pip install autogluon' at the top of the notebook!
try:
    from autogluon.tabular import TabularPredictor
    
    train_meta = pd.DataFrame(oof_master, columns=available_models)
    train_meta['Drafted'] = y
    test_meta = pd.DataFrame(test_master, columns=available_models)
    
    # Train AutoGluon purely on the model predictions (it builds trees on top of trees!)
    predictor = TabularPredictor(label='Drafted', eval_metric='roc_auc').fit(
        train_meta, 
        time_limit=120, # 2 minutes max
        presets='best_quality'
    )
    
    # Get the OOF CV score from AutoGluon
    leaderboard = predictor.leaderboard(train_meta, silent=True)
    ag_score = leaderboard[leaderboard['model'] == 'WeightedEnsemble_L2']['score_val'].values[0] if 'WeightedEnsemble_L2' in leaderboard['model'].values else leaderboard['score_val'].max()
    print(f"AutoGluon Stack CV: {ag_score:.5f}")
    
    ag_test_preds = predictor.predict_proba(test_meta)[1].values
except ImportError:
    print("AutoGluon not installed. Skipping. (Run !pip install autogluon if you want to use it!)")
    ag_score = 0
    ag_test_preds = None


In [ ]:
# Save Final Submission (Using whichever scored highest)
scores = {'Logistic Regression': lr_score, 'Hill Climbing': v6_score, 'Rank Averaging': rank_score, 'AutoGluon': ag_score}
best_method = max(scores, key=scores.get)
print(f"\n--- WINNER: {best_method} with CV {scores[best_method]:.5f} ---")

if best_method == 'Logistic Regression':
    best_test_preds = meta_lr.predict_proba(test_master)[:, 1]
elif best_method == 'Hill Climbing':
    best_test_preds = ensemble_test
elif best_method == 'Rank Averaging':
    best_test_preds = ensemble_rank_test
else:
    best_test_preds = ag_test_preds

submission = pd.DataFrame({'Id': test['Id'], 'Drafted': best_test_preds})
submission.to_csv('submission.csv', index=False)
print("Saved final submission.csv! Download it from the side panel.")
